# LeetCode 2265 — Count Nodes Equal to Average of Subtree bằng Python

Link đề: https://leetcode.com/problems/count-nodes-equal-to-average-of-subtree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def averageOfSubtree(self, root: TreeNode) -> int:
        ...
```

## 1. Mục tiêu học

1. Tự dựng `TreeNode` và hàm **dựng cây từ mảng kiểu LeetCode** (`[4,8,5,0,1,null,6]`) để test.
2. Hiểu **duyệt hậu thứ tự** (post-order): thông tin của một nút (tổng, số nút cây con) được **ghép từ hai con** ⇒ mỗi nút xử lý O(1).
3. So sánh vét cạn O(n²) (tính lại cây con cho mỗi nút) với DFS trả về cặp `(tổng, số nút)` O(n).
4. Cài đặt bản **không đệ quy** (ngăn xếp tường minh / thứ tự BFS đảo ngược) — quan trọng vì cây suy biến 1000 nút **vượt** giới hạn đệ quy mặc định của Python.
5. Dùng đúng phép chia làm tròn xuống `//` cho trung bình.

## 2. Đề bài & ví dụ

Cho gốc `root` của cây nhị phân. Trả về **số nút** mà giá trị của nút **bằng trung bình** các giá trị trong **cây con** của nó.

- Trung bình của `n` số = tổng chia `n`, **làm tròn xuống**.
- Cây con của một nút gồm chính nút đó và mọi hậu duệ.

**Ví dụ 1:** `root = [4,8,5,0,1,null,6]` → `5`:
- nút 4: `(4+8+5+0+1+6)/6 = 24/6 = 4` ✔; nút 5: `(5+6)/2 = 5` ✔; các lá 0, 1, 6 ✔; nút 8: `(8+0+1)/3 = 3 ≠ 8` ✘.

**Ví dụ 2:** `root = [1]` → `1`.

**Ràng buộc:** số nút trong `[1, 1000]`, `0 ≤ Node.val ≤ 1000`.

In [1]:
from typing import Optional, List
from collections import deque
import random, time, sys

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val; self.left = left; self.right = right

def build(vals):
    # dựng cây từ mảng theo thứ tự tầng kiểu LeetCode (None = không có nút)
    if not vals or vals[0] is None: return None
    root = TreeNode(vals[0]); q = deque([root]); i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i]); q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i]); q.append(node.right)
        i += 1
    return root

def to_list(root):
    out = []; q = deque([root])
    while q:
        node = q.popleft()
        if node: out.append(node.val); q.append(node.left); q.append(node.right)
        else: out.append(None)
    while out and out[-1] is None: out.pop()
    return out

examples = [([4, 8, 5, 0, 1, None, 6], 5), ([1], 1)]
for vals, exp in examples:
    print(vals, "->", to_list(build(vals)), "| kỳ vọng", exp)

[4, 8, 5, 0, 1, None, 6] -> [4, 8, 5, 0, 1, None, 6] | kỳ vọng 5
[1] -> [1] | kỳ vọng 1


## 3. Kiến thức nền

### 3.1 Cây nhị phân và cây con

Mỗi nút có tối đa hai con. **Cây con** của nút `v` = `v` + cây con của con trái + cây con của con phải. Định nghĩa đệ quy này là nền tảng của mọi thuật toán DFS trên cây.

In [2]:
def show(root, depth=0, tag="gốc"):
    if root is None: return
    print("    " * depth + f"{tag}: {root.val}")
    show(root.left, depth + 1, "T"); show(root.right, depth + 1, "P")
show(build([4, 8, 5, 0, 1, None, 6]))

gốc: 4
    T: 8
        T: 0
        P: 1
    P: 5
        P: 6


### 3.2 Duyệt hậu thứ tự (post-order): con trái → con phải → nút

Khi xử lý nút, kết quả của hai con **đã có** ⇒ ghép lại được ngay.

In [3]:
def postorder(root, out):
    if root:
        postorder(root.left, out); postorder(root.right, out); out.append(root.val)
    return out
print(postorder(build([4, 8, 5, 0, 1, None, 6]), []))

[0, 1, 8, 6, 5, 4]


### 3.3 Giới hạn đệ quy của Python

Mặc định ~1000 khung. Cây suy biến (mọi nút chỉ có con phải) với 1000 nút cần đệ quy sâu ~1001 ⇒ **có thể** gây `RecursionError`. Ta kiểm chứng ở mục 13; bản nộp nên dùng vòng lặp.

In [4]:
print("giới hạn đệ quy:", sys.getrecursionlimit())

giới hạn đệ quy: 1000


### 3.4 Chia làm tròn xuống `//`

Giá trị không âm ⇒ `s // c` = làm tròn xuống. Không dùng `round(s / c)` (làm tròn gần nhất) hay so sánh `s / c == val` (sai khi trung bình không nguyên: `11 / 2 = 5,5 ≠ 5` nhưng đề làm tròn xuống thành 5).

In [5]:
print(11 // 2, 11 / 2, round(11 / 2), 24 // 6)

5 5.5 6 4


## 4. Ý tưởng nền tảng / trực giác

Với mỗi nút cần **tổng** và **số nút** của cây con. Hai đại lượng này **cộng được** từ dưới lên:

`sum(v) = v.val + sum(trái) + sum(phải)`, `cnt(v) = 1 + cnt(trái) + cnt(phải)` (cây rỗng: `(0, 0)`).

Tính tay ví dụ 1 từ lá lên gốc:

| nút | (sum, cnt) | sum // cnt | bằng val? |
|---|---|---|---|
| 0 (lá) | (0, 1) | 0 | ✔ |
| 1 (lá) | (1, 1) | 1 | ✔ |
| 6 (lá) | (6, 1) | 6 | ✔ |
| 8 | (8 + 0 + 1, 3) = (9, 3) | 3 | ✘ |
| 5 | (5 + 0 + 6, 2) = (11, 2) | 5 | ✔ |
| 4 | (4 + 9 + 11, 6) = (24, 6) | 4 | ✔ |

Đếm ✔: 5. Mọi lá luôn thỏa (trung bình của chính nó).

## 5. Cách 1 — Brute Force / baseline: với mỗi nút, duyệt lại cả cây con

**Ý tưởng:** duyệt mọi nút; với mỗi nút gọi hàm duyệt cây con để tính tổng và số nút.

In [6]:
def subtree_stats(node):
    s = c = 0; stack = [node]
    while stack:
        x = stack.pop()
        if x:
            s += x.val; c += 1; stack.append(x.left); stack.append(x.right)
    return s, c

def brute(root):
    ans = 0; stack = [root]
    while stack:
        v = stack.pop()
        if v:
            s, c = subtree_stats(v)
            ans += (s // c == v.val)
            stack.append(v.left); stack.append(v.right)
    return ans

for vals, exp in examples:
    got = brute(build(vals))
    print(got, "OK" if got == exp else "SAI")

5 OK
1 OK


**Độ phức tạp:** mỗi nút được đếm trong cây con của mọi tổ tiên ⇒ tổng = Σ (độ sâu + 1) ⇒ O(n·h): O(n log n) với cây cân bằng, **O(n²)** với cây suy biến (`n = 1000` ⇒ ~5·10⁵ — vẫn chạy, nhưng đây là cách lãng phí).

**Nút thắt:** cây con của một nút chứa cây con của các con — tính lại từ đầu là thừa. Tính **từ dưới lên** và dùng lại kết quả của con.

In [7]:
def chain(n):
    root = cur = TreeNode(0)
    for i in range(1, n):
        cur.right = TreeNode(i % 7); cur = cur.right
    return root

for n in [300, 1000]:
    r = chain(n); t = time.perf_counter(); brute(r); print(f"cây suy biến {n} nút: brute {(time.perf_counter()-t)*1000:.0f} ms")

cây suy biến 300 nút: brute 3 ms
cây suy biến 1000 nút: brute 38 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** định nghĩa `f(v) = (tổng, số nút)` của cây con `v`. Khi đó `f(v) = (v.val + s_L + s_R, 1 + c_L + c_R)` — chỉ cần kết quả của hai con. Duyệt **hậu thứ tự** tính `f` cho mọi nút trong O(n), và tại mỗi nút kiểm tra `s // c == v.val`.

**Quy tắc:**
- **Nếu** nút rỗng **thì** `(0, 0)`.
- **Ngược lại** ghép hai con, cộng thêm chính nút; **nếu** `s // c == val` **thì** đếm.

**Bất biến (bản lặp):** khi một nút được xử lý (lần "rời" nút), kết quả của cả hai con đã được lưu.

**Ba cách tổ chức không đệ quy:**
1. Ngăn xếp với cờ "đã thăm con" (`(node, visited)`).
2. Lấy thứ tự **tiền thứ tự đảo** (gốc → phải → trái, rồi đảo lại = trái → phải → gốc là hậu thứ tự).
3. Thứ tự **BFS** rồi xử lý **ngược** (con luôn xuất hiện sau cha trong BFS ⇒ duyệt ngược thì con xong trước cha).

### 6.1 Đại lượng nào "ghép được từ con"?

Kỹ thuật post-order chỉ hiệu quả khi đại lượng cần tính **ghép được** từ kết quả của hai con trong O(1): tổng, số nút, max, min, chiều cao, XOR… **Trung bình** thì không ghép trực tiếp được (trung bình của hai trung bình ≠ trung bình chung khi kích thước khác nhau) — nên ta mang theo **(tổng, số nút)** thay vì mang trung bình. **Trung vị** còn tệ hơn: cần cả danh sách giá trị (bài tập 5).

In [8]:
left_vals, right_vals = [1, 2, 3, 10], [4]
avg = lambda a: sum(a) / len(a)
print("trung bình của hai trung bình:", (avg(left_vals) + avg(right_vals)) / 2)
print("trung bình chung             :", avg(left_vals + right_vals))
print("ghép từ (tổng, số nút)       :", (sum(left_vals) + sum(right_vals)) / (len(left_vals) + len(right_vals)))

trung bình của hai trung bình: 4.0
trung bình chung             : 4.0
ghép từ (tổng, số nút)       : 4.0


Bài học: khi một đại lượng không ghép được, hãy tìm **bộ đại lượng ghép được** mà từ đó suy ra nó (ở đây: tổng và số lượng).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS đệ quy trả về `(tổng, số nút)` (cách doocs)

In [9]:
def dfs_recursive(root):
    ans = 0
    def dfs(node):
        nonlocal ans
        if node is None:
            return 0, 0
        ls, lc = dfs(node.left)
        rs, rc = dfs(node.right)
        s, c = ls + rs + node.val, lc + rc + 1
        if s // c == node.val:
            ans += 1
        return s, c
    dfs(root)
    return ans

for vals, exp in examples:
    got = dfs_recursive(build(vals))
    print(got, "OK" if got == exp else "SAI")

5 OK
1 OK


**Độ phức tạp cách 2:** O(n) thời gian, O(h) ngăn xếp đệ quy (h = chiều cao, tới 1000 với cây suy biến).

### 7.2 Cách 3 — Ngăn xếp tường minh với cờ "đã thăm"

In [10]:
def iterative_flag(root):
    ans = 0; stats = {}                      # id(node) -> (sum, cnt)
    stack = [(root, False)]
    while stack:
        node, visited = stack.pop()
        if node is None: continue
        if not visited:
            stack.append((node, True))       # quay lại sau khi xử lý hai con
            stack.append((node.right, False)); stack.append((node.left, False))
        else:
            ls, lc = stats.get(id(node.left), (0, 0)) if node.left else (0, 0)
            rs, rc = stats.get(id(node.right), (0, 0)) if node.right else (0, 0)
            s, c = ls + rs + node.val, lc + rc + 1
            stats[id(node)] = (s, c)
            ans += s // c == node.val
    return ans

for vals, exp in examples:
    got = iterative_flag(build(vals))
    print(got, "OK" if got == exp else "SAI")

5 OK
1 OK


Dùng `id(node)` làm khóa vì `TreeNode` mặc định băm theo danh tính (identity) — an toàn. **Độ phức tạp:** O(n) thời gian, O(n) bộ nhớ.

### 7.3 Cách 4 — Thứ tự BFS rồi xử lý ngược

In [11]:
def bfs_reverse(root):
    order = []; q = deque([root])
    while q:
        node = q.popleft(); order.append(node)
        if node.left: q.append(node.left)
        if node.right: q.append(node.right)
    s = {}; c = {}; ans = 0
    for node in reversed(order):              # con luôn đứng sau cha trong BFS
        ss = node.val + (s[id(node.left)] if node.left else 0) + (s[id(node.right)] if node.right else 0)
        cc = 1 + (c[id(node.left)] if node.left else 0) + (c[id(node.right)] if node.right else 0)
        s[id(node)] = ss; c[id(node)] = cc
        ans += ss // cc == node.val
    return ans

for vals, exp in examples:
    got = bfs_reverse(build(vals))
    print(got, "OK" if got == exp else "SAI")

5 OK
1 OK


**Độ phức tạp cách 4:** O(n) thời gian, O(n) bộ nhớ. Rất dễ viết đúng, không cần cờ.

### 7.4 Cách 5 — Tiền thứ tự đảo (gốc → phải → trái) rồi đảo ngược

Thứ tự "gốc, phải, trái" đảo lại thành "trái, phải, gốc" = hậu thứ tự. Sau đó xử lý theo hậu thứ tự với `dict` kết quả.

In [12]:
def reversed_preorder(root):
    st = [root]; order = []
    while st:
        node = st.pop(); order.append(node)
        if node.left: st.append(node.left)
        if node.right: st.append(node.right)
    res = {None: (0, 0)}; ans = 0
    for node in reversed(order):              # hậu thứ tự
        ls, lc = res[node.left]; rs, rc = res[node.right]
        s, cnt = ls + rs + node.val, lc + rc + 1
        res[node] = (s, cnt); ans += s // cnt == node.val
    return ans

for vals, exp in examples:
    print(reversed_preorder(build(vals)), "OK" if reversed_preorder(build(vals)) == exp else "SAI")

5 OK
1 OK


Mẹo `res = {None: (0, 0)}` cho phép tra trực tiếp `res[node.left]` kể cả khi con rỗng (`None` là khóa hợp lệ). Dùng chính `node` làm khóa (băm theo danh tính).

## 8. TRACE từng bước

### 8.1 Trace cách 2 (đệ quy) trên ví dụ 1 — in theo thứ tự hậu thứ tự

In [13]:
def trace_dfs(root):
    ans = 0
    def dfs(node, depth):
        nonlocal ans
        if node is None: return 0, 0
        ls, lc = dfs(node.left, depth + 1); rs, rc = dfs(node.right, depth + 1)
        s, c = ls + rs + node.val, lc + rc + 1
        ok = s // c == node.val; ans += ok
        print("  " * depth + f"nút {node.val}: trái=({ls},{lc}) phải=({rs},{rc}) -> ({s},{c}), {s}//{c}={s // c} {'✔' if ok else '✘'} (đếm={ans})")
        return s, c
    dfs(root, 0); return ans

print("Kết quả:", trace_dfs(build([4, 8, 5, 0, 1, None, 6])))

    nút 0: trái=(0,0) phải=(0,0) -> (0,1), 0//1=0 ✔ (đếm=1)
    nút 1: trái=(0,0) phải=(0,0) -> (1,1), 1//1=1 ✔ (đếm=2)
  nút 8: trái=(0,1) phải=(1,1) -> (9,3), 9//3=3 ✘ (đếm=2)
    nút 6: trái=(0,0) phải=(0,0) -> (6,1), 6//1=6 ✔ (đếm=3)
  nút 5: trái=(0,0) phải=(6,1) -> (11,2), 11//2=5 ✔ (đếm=4)
nút 4: trái=(9,3) phải=(11,2) -> (24,6), 24//6=4 ✔ (đếm=5)
Kết quả: 5


### 8.2 Trace cách 4 (BFS đảo ngược) — thứ tự xử lý

In [14]:
def trace_bfs(root):
    order = []; q = deque([root])
    while q:
        node = q.popleft(); order.append(node)
        if node.left: q.append(node.left)
        if node.right: q.append(node.right)
    print("thứ tự BFS:", [x.val for x in order])
    print("xử lý ngược:", [x.val for x in reversed(order)], "(mỗi nút được xử lý sau mọi con của nó)")

trace_bfs(build([4, 8, 5, 0, 1, None, 6]))

thứ tự BFS: [4, 8, 5, 0, 1, 6]
xử lý ngược: [6, 1, 0, 5, 8, 4] (mỗi nút được xử lý sau mọi con của nó)


### 8.3 Ví dụ có trung bình **không nguyên**: `[2, 1, 4]`

Nút 2: `(2 + 1 + 4) / 3 = 7/3 = 2,33 → 2` ✔ (làm tròn xuống). Hai lá ✔ ⇒ đáp án 3.

In [15]:
print("Kết quả:", trace_dfs(build([2, 1, 4])))

  nút 1: trái=(0,0) phải=(0,0) -> (1,1), 1//1=1 ✔ (đếm=1)
  nút 4: trái=(0,0) phải=(0,0) -> (4,1), 4//1=4 ✔ (đếm=2)
nút 2: trái=(1,1) phải=(4,1) -> (7,3), 7//3=2 ✔ (đếm=3)
Kết quả: 3


## 9. Code nộp LeetCode

Bản lặp (BFS đảo ngược) — an toàn với cây suy biến 1000 nút.

In [16]:
from collections import deque

class Solution:
    def averageOfSubtree(self, root: TreeNode) -> int:
        order = []
        q = deque([root])
        while q:                                  # thứ tự BFS: cha luôn trước con
            node = q.popleft()
            order.append(node)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        res = {None: (0, 0)}                      # nút -> (tổng, số nút) của cây con
        ans = 0
        for node in reversed(order):              # con luôn được xử lý trước cha
            ls, lc = res[node.left]
            rs, rc = res[node.right]
            s, c = ls + rs + node.val, lc + rc + 1
            res[node] = (s, c)
            if s // c == node.val:
                ans += 1
        return ans

In [17]:
sol = Solution()
for vals, exp in examples:
    got = sol.averageOfSubtree(build(vals))
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

5 | kỳ vọng 5 | OK
1 | kỳ vọng 1 | OK


## 10. Vì sao thuật toán đúng

**Bổ đề (cấu trúc đệ quy).** Với mọi nút `v`: `sum(v) = v.val + sum(v.left) + sum(v.right)` và `cnt(v) = 1 + cnt(v.left) + cnt(v.right)`, với quy ước cây rỗng `(0, 0)`. Đúng vì cây con của `v` là hợp **rời** của `{v}`, cây con trái và cây con phải.

**Thứ tự xử lý.** Trong thứ tự BFS, mỗi con được đưa vào hàng đợi **sau khi** cha được lấy ra ⇒ con xuất hiện **sau** cha trong `order`. Duyệt ngược ⇒ con xử lý **trước** cha ⇒ khi xử lý `v`, `res[v.left]` và `res[v.right]` đã đúng (quy nạp theo chiều cao).

**Đếm đúng.** Mỗi nút được xử lý đúng một lần và kiểm tra đúng điều kiện `⌊sum/cnt⌋ = val` của đề. ∎

## 11. Phân tích Time / Space Complexity

`n ≤ 1000`, `h` = chiều cao cây.

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt lại cây con mỗi nút | O(n·h) (O(n²) worst) | O(h) |
| 2. DFS đệ quy | O(n) | O(h) ngăn xếp gọi hàm |
| 3. Ngăn xếp + cờ | O(n) | O(n) |
| 4. BFS đảo ngược | O(n) | O(n) |
| 5. Tiền thứ tự đảo | O(n) | O(n) |

In [18]:
def random_tree(n, rng, maxv=1000):
    nodes = [TreeNode(rng.randint(0, maxv)) for _ in range(n)]
    for i in range(1, n):
        while True:
            par = nodes[rng.randrange(i)]
            if par.left is None and rng.random() < 0.5: par.left = nodes[i]; break
            if par.right is None: par.right = nodes[i]; break
    return nodes[0]

rng = random.Random(2265)
for name_t, tree in [("ngẫu nhiên 1000", random_tree(1000, rng)), ("suy biến 990", chain(990))]:
    for name, f in [("brute", brute), ("iterative_flag", iterative_flag), ("bfs_reverse", bfs_reverse),
                    ("reversed_preorder", reversed_preorder), ("Solution", Solution().averageOfSubtree)]:
        t = time.perf_counter(); r = f(tree); dt = time.perf_counter() - t
        print(f"{name_t:<16} {name:<18} -> {r}  {dt*1000:6.1f} ms")

ngẫu nhiên 1000  brute              -> 358     2.6 ms
ngẫu nhiên 1000  iterative_flag     -> 358     0.6 ms
ngẫu nhiên 1000  bfs_reverse        -> 358     0.6 ms
ngẫu nhiên 1000  reversed_preorder  -> 358     0.4 ms
ngẫu nhiên 1000  Solution           -> 358     0.3 ms


suy biến 990     brute              -> 284    60.8 ms
suy biến 990     iterative_flag     -> 284     0.6 ms
suy biến 990     bfs_reverse        -> 284     0.5 ms
suy biến 990     reversed_preorder  -> 284     0.3 ms
suy biến 990     Solution           -> 284     0.3 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [19]:
ALL = {"brute": brute, "dfs_recursive": dfs_recursive, "iterative_flag": iterative_flag,
       "bfs_reverse": bfs_reverse, "reversed_preorder": reversed_preorder, "Solution": Solution().averageOfSubtree}
edge = examples + [
    ([0], 1),
    ([2, 1, 4], 3),                     # 7 // 3 = 2
    ([1, 2, 3], 2),                     # gốc: 6 // 3 = 2 != 1
    ([5, 5, 5, 5, 5], 5),               # mọi nút bằng nhau
    ([0, None, 0, None, 0], 3),
    ([1000, 0, None, 0], 2),            # gốc 1000//3=333; nút 0: 0//2=0 ✔; lá ✔
    ([3, 1, 5, 0, 2, 4, 6], 7),         # mỗi cây con: trung bình làm tròn xuống = gốc của nó
]
for vals, exp in edge:
    for name, f in ALL.items():
        assert f(build(vals)) == exp, (name, vals, f(build(vals)), exp)
print("Edge case OK:", len(edge))

Edge case OK: 9


Kiểm tay `[3, 1, 5, 0, 2, 4, 6]` (cây tìm kiếm cân bằng của 0..6): nút 1: `(1+0+2)/3 = 1` ✔; nút 5: `(5+4+6)/3 = 5` ✔; gốc: `21/7 = 3` ✔; 4 lá ✔ ⇒ 7.

### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [20]:
rng = random.Random(42)
for _ in range(500):
    n = rng.randint(1, 40); mv = rng.choice([0, 2, 5, 1000])
    tree = random_tree(n, rng, mv)
    ref = brute(tree)
    for name, f in ALL.items():
        assert f(tree) == ref, (name, to_list(tree))
print("500 cây ngẫu nhiên OK")

500 cây ngẫu nhiên OK


### 12.3 Kiểm tra `build`/`to_list` khứ hồi và tính chất "mọi lá đều được đếm"

In [21]:
def leaves(root):
    st = [root]; c = 0
    while st:
        x = st.pop()
        if x:
            if not x.left and not x.right: c += 1
            st.append(x.left); st.append(x.right)
    return c

rng = random.Random(7)
for _ in range(300):
    t = random_tree(rng.randint(1, 30), rng)
    assert to_list(build(to_list(t))) == to_list(t)
    assert Solution().averageOfSubtree(t) >= leaves(t)
print("khứ hồi + tính chất lá OK")

khứ hồi + tính chất lá OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So sánh với trung bình **thực** (`s / c == val`)

In [22]:
def wrong_float(root):
    ans = 0
    def dfs(node):
        nonlocal ans
        if not node: return 0, 0
        a, b = dfs(node.left); c, d = dfs(node.right)
        s, k = a + c + node.val, b + d + 1
        ans += s / k == node.val                          # SAI: không làm tròn xuống
        return s, k
    dfs(root); return ans

print("bản sai :", wrong_float(build([4, 8, 5, 0, 1, None, 6])), "| bản đúng:", Solution().averageOfSubtree(build([4, 8, 5, 0, 1, None, 6])))

bản sai : 4 | bản đúng: 5


**Vì sao sai:** nút 5 có trung bình `11/2 = 5,5`; đề làm tròn xuống thành 5 ⇒ được đếm. **Sửa:** `s // c`.

### Lỗi 2 — Chỉ tính trung bình với **hai con trực tiếp** (không phải cả cây con)

In [23]:
def wrong_children_only(root):
    ans = 0; st = [root]
    while st:
        v = st.pop()
        if v:
            vals = [v.val] + [x.val for x in (v.left, v.right) if x]
            ans += sum(vals) // len(vals) == v.val          # SAI: bỏ qua cháu, chắt...
            st.append(v.left); st.append(v.right)
    return ans

print("bản sai :", wrong_children_only(build([4, 8, 5, 0, 1, None, 6])), "| bản đúng:", Solution().averageOfSubtree(build([4, 8, 5, 0, 1, None, 6])))

bản sai : 4 | bản đúng: 5


**Vì sao sai:** gốc 4 với hai con `8, 5`: `17 // 3 = 5 ≠ 4`; nhưng cả cây con cho `24 // 6 = 4`. **Sửa:** cộng dồn **toàn bộ** cây con (post-order).

### Lỗi 3 — Quên đếm chính nút vào `cnt` (hoặc vào `sum`)

In [24]:
def wrong_count_self(root):
    ans = 0
    def dfs(node):
        nonlocal ans
        if not node: return 0, 0
        a, b = dfs(node.left); c, d = dfs(node.right)
        s, k = a + c + node.val, b + d                     # SAI: quên +1
        if k and s // k == node.val: ans += 1
        return s, k
    dfs(root); return ans

print("bản sai :", wrong_count_self(build([4, 8, 5, 0, 1, None, 6])), "| bản đúng:", Solution().averageOfSubtree(build([4, 8, 5, 0, 1, None, 6])))

bản sai : 0 | bản đúng: 5


**Sửa:** `cnt = 1 + cnt_trái + cnt_phải`.

### Lỗi 4 — Đệ quy trên cây suy biến 1000 nút (`RecursionError`)

In [25]:
deep = chain(1000)
old = sys.getrecursionlimit()
try:
    sys.setrecursionlimit(1000)
    print("đệ quy:", dfs_recursive(deep))
except RecursionError as e:
    print("đệ quy -> RecursionError:", str(e)[:45])
finally:
    sys.setrecursionlimit(old)
print("bản lặp:", Solution().averageOfSubtree(deep))

đệ quy -> RecursionError: maximum recursion depth exceeded
bản lặp: 144


**Vì sao sai:** độ sâu gọi hàm ≈ số nút trên đường dài nhất + khung của notebook/kernel ⇒ vượt 1000. Trên LeetCode bản đệ quy thường vẫn qua vì họ nâng giới hạn, nhưng đừng dựa vào điều đó. **Sửa:** dùng bản lặp (cách 3/4/5), hoặc `sys.setrecursionlimit(10**4)` một cách có ý thức.

### Lỗi 5 — Dùng biến toàn cục / thuộc tính lớp để đếm mà không reset giữa các lần gọi

In [26]:
class WrongCounter:
    count = 0                                             # SAI: thuộc tính lớp, dùng chung giữa các lần gọi
    def averageOfSubtree(self, root):
        def dfs(node):
            if not node: return 0, 0
            a, b = dfs(node.left); c, d = dfs(node.right)
            s, k = a + c + node.val, b + d + 1
            if s // k == node.val: WrongCounter.count += 1
            return s, k
        dfs(root); return WrongCounter.count

w = WrongCounter()
print("lần 1:", w.averageOfSubtree(build([1])), "| lần 2:", w.averageOfSubtree(build([1])), "| đúng mỗi lần: 1")

lần 1: 1 | lần 2: 2 | đúng mỗi lần: 1


**Vì sao sai:** LeetCode chạy nhiều test trên **cùng một tiến trình**; bộ đếm ở mức lớp/toàn cục cộng dồn giữa các test. **Sửa:** biến cục bộ (`nonlocal`) hoặc thuộc tính `self` khởi tạo trong hàm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Thông tin của cây con" (tổng, số nút, max, chiều cao…) ⇒ **post-order**, mỗi nút trả về một bộ giá trị ghép từ con.
- Trả về **tuple** từ DFS thay vì tính nhiều lần; đại lượng không ghép được (trung bình) ⇒ mang theo các thành phần ghép được (tổng, số lượng).
- Cây tới 10³–10⁵ nút, có thể suy biến ⇒ ưu tiên **bản lặp** (BFS đảo / ngăn xếp).
- Trung bình "làm tròn xuống" ⇒ `//`.
- Bài cùng họ: 1973 Count Nodes Equal to Sum of Descendants, 508 Most Frequent Subtree Sum, 1120 Maximum Average Subtree, 563 Binary Tree Tilt, 2049.

## 15. Biến thể của bài

### 15.1 Biến thể A — LC 1120: **trung bình cây con lớn nhất** (số thực)

In [27]:
def maximum_average_subtree(root):
    order = []; q = deque([root])
    while q:
        x = q.popleft(); order.append(x)
        if x.left: q.append(x.left)
        if x.right: q.append(x.right)
    res = {None: (0, 0)}; best = 0.0
    for x in reversed(order):
        ls, lc = res[x.left]; rs, rc = res[x.right]
        s, c = ls + rs + x.val, lc + rc + 1; res[x] = (s, c)
        best = max(best, s / c)
    return best

print(maximum_average_subtree(build([5, 6, 1])), maximum_average_subtree(build([0, None, 1])))

6.0 1.0


### 15.2 Biến thể B — LC 1973: nút có giá trị bằng **tổng các hậu duệ** (không gồm chính nó)

In [28]:
def equal_to_descendants(root):
    order = []; q = deque([root])
    while q:
        x = q.popleft(); order.append(x)
        if x.left: q.append(x.left)
        if x.right: q.append(x.right)
    sub = {None: 0}; ans = 0
    for x in reversed(order):
        desc = sub[x.left] + sub[x.right]
        ans += desc == x.val
        sub[x] = desc + x.val
    return ans

print(equal_to_descendants(build([10, 3, 4, 2, 1])), equal_to_descendants(build([2, 3, None, 2, None])), equal_to_descendants(build([0])))

2 0 1


### 15.3 Biến thể C — Trả về **danh sách giá trị** các nút thỏa (theo thứ tự BFS) và kiểm tra bằng brute force

In [29]:
def matching_nodes(root):
    order = []; q = deque([root])
    while q:
        x = q.popleft(); order.append(x)
        if x.left: q.append(x.left)
        if x.right: q.append(x.right)
    res = {None: (0, 0)}; ok = set()
    for x in reversed(order):
        ls, lc = res[x.left]; rs, rc = res[x.right]
        s, c = ls + rs + x.val, lc + rc + 1; res[x] = (s, c)
        if s // c == x.val: ok.add(x)
    return [x.val for x in order if x in ok]

print(matching_nodes(build([4, 8, 5, 0, 1, None, 6])))

[4, 5, 0, 1, 6]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt lại cây con | tính lại mỗi nút | O(n·h) | O(h) | kiểm chứng |
| 2. DFS đệ quy | trả `(sum, cnt)` | O(n) | O(h) | ngắn, cây không quá sâu |
| 3. Ngăn xếp + cờ | post-order lặp | O(n) | O(n) | luyện post-order lặp |
| 4. BFS đảo ngược | con sau cha trong BFS | O(n) | O(n) | **mặc định**, an toàn |
| 5. Tiền thứ tự đảo | gốc-phải-trái đảo = hậu thứ tự | O(n) | O(n) | gọn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `averageOfSubtree(build([6, 2, 9]))` và `averageOfSubtree(build([1, None, 3, None, 5]))`?

**Bài 2 (tính tay):** Với cây `[10, 2, 18, None, None, 14, 22]`, lập bảng `(sum, cnt, sum//cnt)` cho mọi nút.

**Bài 3 (điền TODO):** Viết `bt3_todo(root)` trả về `dict` *giá trị nút → (sum, cnt)* (giả sử giá trị phân biệt) bằng post-order lặp.

**Bài 4 (tự viết lại):** Viết `bt4_todo(root)` cài cách 3 (ngăn xếp + cờ) không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(root)` trả về số nút mà giá trị bằng **trung vị** (median, phần tử thứ `⌊(c−1)/2⌋` sau khi sắp xếp) của cây con — với n ≤ 300, dùng gộp danh sách đã sắp xếp từ hai con.

In [30]:
def bt3_todo(root):
    # TODO: dict val -> (sum, cnt)
    pass

def bt4_todo(root):
    # TODO: ngăn xếp + cờ
    pass

def bt5_todo(root):
    # TODO: đếm nút bằng trung vị cây con
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: lá luôn ✔. Mức 2: gốc 6: `17 // 3 = 5`. Mức 3: `2`; cây hai: nút 5 ✔, nút 3: `8 // 2 = 4` ✘, nút 1: `9 // 3 = 3` ✘ ⇒ `1`.

**Bài 2.** Mức 1: lá 2, 14, 22. Mức 2: nút 18: `(54, 3, 18)` ✔. Mức 3: gốc: `(66, 5, 13)` ✘ ⇒ đáp án 4.

**Bài 3.** Mức 1: BFS đảo ngược. Mức 2: lưu `res[node]`. Mức 3: đổi khóa thành `node.val`.

**Bài 4.** Mức 1: đẩy `(node, False)`. Mức 2: lần đầu gặp: đẩy lại `(node, True)` rồi hai con. Mức 3: lần hai: ghép kết quả con.

**Bài 5.** Mức 1: mỗi nút cần danh sách giá trị cây con đã sắp xếp. Mức 2: gộp (merge) hai danh sách con rồi chèn giá trị nút (`bisect.insort`). Mức 3: trung vị ở chỉ số `(c − 1) // 2`; O(n²) với n ≤ 300 là ổn.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `2`, `1`. **Bài 2:** đáp án `4` (bảng ở cell dưới).

In [31]:
from bisect import insort
from heapq import merge
assert Solution().averageOfSubtree(build([6, 2, 9])) == 2
assert Solution().averageOfSubtree(build([1, None, 3, None, 5])) == 1
assert Solution().averageOfSubtree(build([10, 2, 18, None, None, 14, 22])) == 4

def bt3(root):
    order = []; q = deque([root])
    while q:
        x = q.popleft(); order.append(x)
        if x.left: q.append(x.left)
        if x.right: q.append(x.right)
    res = {None: (0, 0)}
    for x in reversed(order):
        ls, lc = res[x.left]; rs, rc = res[x.right]; res[x] = (ls + rs + x.val, lc + rc + 1)
    return {x.val: res[x] for x in order}

def bt4(root):
    return iterative_flag(root)

def bt5(root):
    order = []; q = deque([root])
    while q:
        x = q.popleft(); order.append(x)
        if x.left: q.append(x.left)
        if x.right: q.append(x.right)
    lst = {None: []}; ans = 0
    for x in reversed(order):
        cur = list(merge(lst[x.left], lst[x.right])); insort(cur, x.val)
        lst[x] = cur; ans += cur[(len(cur) - 1) // 2] == x.val
    return ans

def bt5_brute(root):
    ans = 0; st = [root]
    while st:
        v = st.pop()
        if v:
            vals = []; s2 = [v]
            while s2:
                y = s2.pop()
                if y: vals.append(y.val); s2.append(y.left); s2.append(y.right)
            vals.sort(); ans += vals[(len(vals) - 1) // 2] == v.val
            st.append(v.left); st.append(v.right)
    return ans

print("Bài 2:", {k: (s, c, s // c) for k, (s, c) in bt3(build([10, 2, 18, None, None, 14, 22])).items()})
rng = random.Random(1965)
for _ in range(300):
    t = random_tree(rng.randint(1, 30), rng, rng.choice([3, 50]))
    assert bt4(t) == brute(t)
    assert bt5(t) == bt5_brute(t)
# bt3 với giá trị phân biệt
vals = random.Random(3).sample(range(100), 15)
t = build(vals)
d = bt3(t)
assert sum(1 for v, (s, c) in d.items() if s // c == v) == brute(t)
print("Đáp án bài tập: tất cả assert OK")

Bài 2: {10: (66, 5, 13), 2: (2, 1, 2), 18: (54, 3, 18), 14: (14, 1, 14), 22: (22, 1, 22)}
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
order, q = [], deque([root])
while q:                                   # BFS: cha trước con
    x = q.popleft(); order.append(x)
    if x.left: q.append(x.left)
    if x.right: q.append(x.right)
res, ans = {None: (0, 0)}, 0
for x in reversed(order):                  # con trước cha
    (ls, lc), (rs, rc) = res[x.left], res[x.right]
    s, c = ls + rs + x.val, lc + rc + 1
    res[x] = (s, c); ans += s // c == x.val
return ans
```

**Checklist:**
- [ ] Tổng và số nút của **cả** cây con (gồm chính nút).
- [ ] Trung bình làm tròn xuống `//`.
- [ ] Cây suy biến 1000 nút: dùng bản lặp hoặc nâng giới hạn đệ quy.
- [ ] Bộ đếm là biến cục bộ (không dùng chung giữa các test).
- [ ] Đã thử: một nút, trung bình không nguyên, mọi nút bằng nhau.